# Step 3 · Model 4: Gradient boosting + blurb score (deep learning)

**Goal:** test whether *what the blurb says* adds anything once we already know the category, goal, length and prep time.

**How, in four moves (the plan's Model 4 rows):**
1. A pre-trained language model (`all-MiniLM-L6-v2`, a small BERT-family sentence encoder, L12 transfer learning) turns every blurb into 384 numbers (an embedding).
2. The **blurb grader**, a logistic regression on the **2016–18** embeddings, learns which blurb meanings tend to get funded.
3. The grader scores every 2019–25 blurb, giving one new column: `blurb_score`.
4. Boosting trains on 2019–23 with the 191 columns **+ `blurb_score`**, then is graded on 2024.

**Two comparisons with Model 3, both reported:**
- **Clean A/B:** Model 3's exact settings, the only difference is the extra column. Answers "does the blurb help?"
- **Honest tuning:** settings chosen on a 2023 inner holdout for *both* models, as Model 3's README recommended. 2024 influences no choice, so no tuning-luck caveat.

**Team rules followed:** launch-day inputs only, no `post_` columns. The grader learns only from 2016–18, which Models 1–3 never saw. From the 2025 test file **only `id` and `blurb` are loaded** (to compute `blurb_score` for the final run); its answer columns are never read.

⚠️ **Runtime ≈ 15 min on a T4 GPU** (Runtime → Change runtime type → T4 GPU). On CPU the embedding step alone takes ~30–40 min. Embeddings are cached to `/content/split/emb_*.npy`, so re-running is fast.

**Before running:** in Colab, unzip `split.zip` to `/content/split`; in VS Code, put the split files in the same folder as this notebook. Either way, also add `baseline_val.parquet`, `model2_val.parquet` and `model3_val.parquet` to that folder.

In [1]:
%pip -q install sentence-transformers pyarrow scikit-learn


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### PIPELINE — Load inputs, answers and the earlier models' scores
Same row-alignment check as Model 2: if `X_*` and `slice_*` rows were offset, every campaign would be graded against someone else's answer and nothing would warn us.

In [2]:
import os, time, warnings
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

BASE = "/content/split"
if not os.path.isdir(BASE):
    BASE = "."

X_train = pd.read_parquet(f"{BASE}/X_train.parquet")
X_val   = pd.read_parquet(f"{BASE}/X_val.parquet")
text    = pd.read_parquet(f"{BASE}/slice_text.parquet", columns=["id", "year", "blurb", "success"])
train   = pd.read_parquet(f"{BASE}/slice_train.parquet")
val     = pd.read_parquet(f"{BASE}/slice_val.parquet")
test_bl = pd.read_parquet(f"{BASE}/slice_test.parquet", columns=["id", "blurb"])   # NO answers loaded

y_text  = text["success"].to_numpy().astype(int)
y_train = train["success"].to_numpy().astype(int)
y_val   = val["success"].to_numpy().astype(int)
nm_val  = val["near_miss"].to_numpy().astype(int)

# row alignment: checked, not assumed
for X, s, name in [(X_train, train, "train"), (X_val, val, "val")]:
    assert len(X) == len(s), name
    assert np.allclose(X["log_goal_usd"].to_numpy(float), s["log_goal_usd"].to_numpy(float)), name
    assert np.allclose(X["blurb_len"].to_numpy(float),
                       s["blurb"].fillna("").astype(str).str.len().to_numpy(float)), name

# earlier models, on identical rows
prev = {}
for m, col in [("baseline_val", "score_model1"), ("model2_val", "score_model2"), ("model3_val", "score_model3")]:
    d = pd.read_parquet(f"{BASE}/{m}.parquet")
    assert (d["id"].to_numpy() == val["id"].to_numpy()).all(), m
    prev[col] = d[col].to_numpy(float)
score_m1, score_m2, score_m3 = prev["score_model1"], prev["score_model2"], prev["score_model3"]
AUC_M3 = roc_auc_score(y_val, score_m3)

print(f"text  {len(text):>7,}   train {len(train):>7,}   val {len(val):>7,}   test blurbs {len(test_bl):>7,}")
print(f"alignment checks passed\nModel 3 AUC on 2024 = {AUC_M3:.4f}   <- the bar to beat")

text  100,240   train 119,297   val  26,739   test blurbs  26,887
alignment checks passed
Model 3 AUC on 2024 = 0.8603   <- the bar to beat


### PIPELINE — Turn every blurb into an embedding
A pre-trained sentence encoder maps each blurb to 384 numbers, so blurbs with similar *meaning* land close together ("A cozy deck-building game for 2–4 players" sits near "A cooperative card game for families"), even with no words in common. We use the encoder **frozen** (no fine-tuning): L14's "frozen backbone + small head", with the grader below as the head.

Embeddings are normalized to length 1 and cached, so this cell only runs once per session.

In [3]:
import torch
from sentence_transformers import SentenceTransformer

if torch.cuda.is_available():
    DEVICE = "cuda"                      # Colab GPU
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"                       # Apple Silicon Mac (VS Code locally)
else:
    DEVICE = "cpu"
print("device:", DEVICE, "" if DEVICE != "cpu" else "  <- slow (~30-40 min). Colab: Runtime > Change runtime type > T4 GPU")
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
encoder = SentenceTransformer(EMB_MODEL, device=DEVICE)

def embed(blurbs, name):
    path = f"{BASE}/emb_{name}.npy"
    if os.path.exists(path):
        E = np.load(path)
        if len(E) == len(blurbs):
            print(f"{name:>5}: loaded cached {E.shape}")
            return E
    t = time.time()
    E = encoder.encode(blurbs.fillna("").astype(str).tolist(), batch_size=256,
                       show_progress_bar=True, normalize_embeddings=True,
                       convert_to_numpy=True).astype(np.float32)
    np.save(path, E)
    print(f"{name:>5}: {E.shape} in {time.time() - t:.0f}s")
    return E

E_text  = embed(text["blurb"],    "text")
E_train = embed(train["blurb"],   "train")
E_val   = embed(val["blurb"],     "val")
E_test  = embed(test_bl["blurb"], "test")

W1115 17:16:57.871000 94741 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


device: mps 


Batches: 100%|██████████| 392/392 [01:10<00:00,  5.57it/s]


 text: (100240, 384) in 71s


Batches: 100%|██████████| 467/467 [01:46<00:00,  4.40it/s]


train: (119297, 384) in 107s


Batches: 100%|██████████| 105/105 [00:27<00:00,  3.85it/s]


  val: (26739, 384) in 27s


Batches: 100%|██████████| 106/106 [00:25<00:00,  4.10it/s]

 test: (26887, 384) in 26s


### PIPELINE — Train the blurb grader on 2016–18 only
A logistic regression on the 384 embedding numbers, predicting `success`. Its one setting, `C` (how hard large weights are taxed), is chosen **inside the text years**: fit on 2016–17, check on 2018, then refit on all of 2016–18. No 2019+ campaign touches the grader.

In [4]:
fit_rows = text["year"].between(2016, 2017).to_numpy()
chk_rows = (text["year"] == 2018).to_numpy()

rows = []
for C in [0.01, 0.1, 1.0, 10.0]:
    g = LogisticRegression(C=C, max_iter=3000)
    g.fit(E_text[fit_rows], y_text[fit_rows])
    rows.append({"C": C, "AUC on 2018": roc_auc_score(y_text[chk_rows], g.predict_proba(E_text[chk_rows])[:, 1])})
grid_grader = pd.DataFrame(rows)
print(grid_grader.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

BEST_C = float(grid_grader.loc[grid_grader["AUC on 2018"].idxmax(), "C"])
grader = LogisticRegression(C=BEST_C, max_iter=3000).fit(E_text, y_text)
print(f"\nchosen C = {BEST_C}  -> refit on all {len(text):,} text-year blurbs")

      C  AUC on 2018
 0.0100       0.7351
 0.1000       0.7436
 1.0000       0.7442
10.0000       0.7438

chosen C = 1.0  -> refit on all 100,240 text-year blurbs


### PIPELINE — Score every 2019–25 blurb
`blurb_score` = the grader's chance of success from the blurb alone. Expect its average to run **low** on later years: the grader learned in 2016–18 when 54% succeeded, versus ~73% after. That doesn't matter here, because boosting only uses the score's *order* to split, and re-learns what each level means from 2019–23.

In [5]:
bs_train = grader.predict_proba(E_train)[:, 1]
bs_val   = grader.predict_proba(E_val)[:, 1]
bs_test  = grader.predict_proba(E_test)[:, 1]      # for the final run only; no answers here

print(f"{'slice':<10}{'mean blurb_score':>18}{'actual success':>16}")
print(f"{'2016-18':<10}{grader.predict_proba(E_text)[:, 1].mean():>18.1%}{y_text.mean():>16.1%}")
print(f"{'2019-23':<10}{bs_train.mean():>18.1%}{y_train.mean():>16.1%}")
print(f"{'2024':<10}{bs_val.mean():>18.1%}{y_val.mean():>16.1%}")
print(f"{'2025':<10}{bs_test.mean():>18.1%}{'(not read)':>16}")

AUC_BLURB = roc_auc_score(y_val, bs_val)
print(f"\nBlurb grader ALONE on 2024: AUC {AUC_BLURB:.4f}   (Model 1 category lookup: {roc_auc_score(y_val, score_m1):.4f})")

slice       mean blurb_score  actual success
2016-18                54.2%           54.2%
2019-23                61.1%           73.5%
2024                   64.9%           72.2%
2025                   65.3%      (not read)

Blurb grader ALONE on 2024: AUC 0.7486   (Model 1 category lookup: 0.8067)


### EXPLORATION — Is the blurb just repeating the category?
A blurb that says "board game" mostly tells the grader the category, which Models 1–3 already know. The real question is whether the blurb ranks campaigns **inside** a category. Model 1 scores 0.500 there by construction; Models 2–3 reached ~0.70–0.82 using goal, length and prep time.

In [6]:
cats = ["Games / Tabletop Games", "Technology / Apps", "Film & Video / Documentary",
        "Publishing / Children's Books", "other"]
print(f"{'subcategory':<32}{'campaigns':>10}{'blurb alone':>13}{'Model 3':>10}")
for c in cats:
    m = (val["subcategory"] == c).to_numpy()
    if m.sum() == 0 or len(np.unique(y_val[m])) < 2:
        continue
    print(f"{c:<32}{m.sum():>10,}{roc_auc_score(y_val[m], bs_val[m]):>13.3f}{roc_auc_score(y_val[m], score_m3[m]):>10.3f}")
print(f"\ncorrelation of blurb_score with Model 3's score on 2024: {np.corrcoef(bs_val, score_m3)[0, 1]:.2f}")

subcategory                      campaigns  blurb alone   Model 3
Games / Tabletop Games               4,907        0.661     0.763
Technology / Apps                      606        0.501     0.811
Film & Video / Documentary             328        0.615     0.712
Publishing / Children's Books          422        0.558     0.676
other                                3,937        0.661     0.774

correlation of blurb_score with Model 3's score on 2024: 0.49


### PIPELINE — Clean A/B: Model 3's exact settings, plus one column
Same data, same settings (`max_depth=None`, `learning_rate=0.1`, 100 trees, `random_state=1`), the only difference is `blurb_score`. Model 3 is refit here first, so both sides come from identical code; boosting is deterministic, so it should reproduce `model3_val.parquet` exactly.

In [7]:
X4_train = X_train.assign(blurb_score=bs_train)
X4_val   = X_val.assign(blurb_score=bs_val)

def hgb(depth, lr, n, warm=False):
    return HistGradientBoostingClassifier(max_depth=depth, learning_rate=lr, max_iter=n,
                                          early_stopping=False, warm_start=warm, random_state=1)

t = time.time()
m3_ab = hgb(None, 0.1, 100).fit(X_train, y_train)
m4_ab = hgb(None, 0.1, 100).fit(X4_train, y_train)
s3_ab = m3_ab.predict_proba(X_val)[:, 1]
s4_ab = m4_ab.predict_proba(X4_val)[:, 1]
print(f"({time.time() - t:.0f}s)")
print(f"Model 3 refit here : AUC {roc_auc_score(y_val, s3_ab):.4f}   (saved file: {AUC_M3:.4f}, max score diff {np.abs(s3_ab - score_m3).max():.1e})")
print(f"Model 4, same setup: AUC {roc_auc_score(y_val, s4_ab):.4f}   gain {roc_auc_score(y_val, s4_ab) - roc_auc_score(y_val, s3_ab):+.4f}")

(71s)
Model 3 refit here : AUC 0.8603   (saved file: 0.8603, max score diff 2.2e-16)
Model 4, same setup: AUC 0.8631   gain +0.0028


### PIPELINE — Honest tuning on a 2023 inner holdout (both models)
Model 3's README measured +0.003 of optimism from choosing settings on 2024 and recommended this for Model 4. Fit on 2019–22, choose settings on 2023, refit on all of 2019–23, then score 2024, which now influenced nothing. Doing it for **both** feature sets makes the comparison honest-vs-honest.

Grid: `max_depth` ∈ {4, None} × 100/200/400/800 trees at `learning_rate=0.1`. These cover both of Model 3's winners (2024 picked `None`/100, 2023 picked `4`/800). `warm_start` adds trees instead of refitting, so 16 settings cost 4 fits.

In [8]:
yr = train["year"].to_numpy()
inner_fit, inner_chk = yr <= 2022, yr == 2023
DEPTHS, LR, CHECKPOINTS = [4, None], 0.1, [100, 200, 400, 800]

def inner_grid(Xtr, label):
    rows = []
    for d in DEPTHS:
        m = hgb(d, LR, CHECKPOINTS[0], warm=True)
        for n in CHECKPOINTS:
            m.set_params(max_iter=n).fit(Xtr[inner_fit], y_train[inner_fit])
            rows.append({"features": label, "max_depth": str(d), "learning_rate": LR, "max_iter": n,
                         "AUC_2023": roc_auc_score(y_train[inner_chk], m.predict_proba(Xtr[inner_chk])[:, 1])})
    return pd.DataFrame(rows)

t = time.time()
grid = pd.concat([inner_grid(X_train, "Model 3 (191 cols)"), inner_grid(X4_train, "Model 4 (+blurb_score)")])
print(f"({time.time() - t:.0f}s)\n")
print(grid.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

def refit_best(label, Xtr, Xva):
    b = grid[grid["features"] == label].sort_values("AUC_2023", ascending=False).iloc[0]
    d = None if b["max_depth"] == "None" else int(b["max_depth"])
    m = hgb(d, float(b["learning_rate"]), int(b["max_iter"])).fit(Xtr, y_train)
    return m, m.predict_proba(Xva)[:, 1], f"depth={b['max_depth']}, lr={b['learning_rate']}, {int(b['max_iter'])} trees"

m3_h, s3_h, set3 = refit_best("Model 3 (191 cols)", X_train, X_val)
model4, score_m4, set4 = refit_best("Model 4 (+blurb_score)", X4_train, X4_val)
AUC_M3H, AUC_M4 = roc_auc_score(y_val, s3_h), roc_auc_score(y_val, score_m4)
print(f"\nModel 3, tuned on 2023 ({set3}): AUC on 2024 {AUC_M3H:.4f}   (README: 0.8571)")
print(f"Model 4, tuned on 2023 ({set4}): AUC on 2024 {AUC_M4:.4f}   gain {AUC_M4 - AUC_M3H:+.4f}")

(370s)

              features max_depth  learning_rate  max_iter  AUC_2023
    Model 3 (191 cols)         4         0.1000       100    0.9083
    Model 3 (191 cols)         4         0.1000       200    0.9155
    Model 3 (191 cols)         4         0.1000       400    0.9198
    Model 3 (191 cols)         4         0.1000       800    0.9207
    Model 3 (191 cols)      None         0.1000       100    0.9201
    Model 3 (191 cols)      None         0.1000       200    0.9206
    Model 3 (191 cols)      None         0.1000       400    0.9198
    Model 3 (191 cols)      None         0.1000       800    0.9183
Model 4 (+blurb_score)         4         0.1000       100    0.9124
Model 4 (+blurb_score)         4         0.1000       200    0.9188
Model 4 (+blurb_score)         4         0.1000       400    0.9230
Model 4 (+blurb_score)         4         0.1000       800    0.9240
Model 4 (+blurb_score)      None         0.1000       100    0.9230
Model 4 (+blurb_score)      None        

### EXPLORATION — Is the gain real, or noise?
Same test as Model 3: resample the 26,739 validation campaigns 200 times and recompute both AUCs on each resample.

In [9]:
rng = np.random.default_rng(0)
n = len(y_val)
pairs = {"Model 4 - Model 3, same settings (A/B)": (s4_ab, s3_ab),
         "Model 4 - Model 3, both tuned on 2023":  (score_m4, s3_h),
         "Model 4 (2023-tuned) - Model 3 shipped": (score_m4, score_m3)}
diffs = {k: [] for k in pairs}
for _ in range(200):
    i = rng.integers(0, n, n)
    if y_val[i].min() == y_val[i].max():
        continue
    for k, (a, b) in pairs.items():
        diffs[k].append(roc_auc_score(y_val[i], a[i]) - roc_auc_score(y_val[i], b[i]))

print(f"{'comparison':<42}{'mean':>9}{'95% CI':>22}{'better in':>11}")
for k, v in diffs.items():
    v = np.array(v)
    print(f"{k:<42}{v.mean():>+9.4f}   [{np.percentile(v, 2.5):+.4f}, {np.percentile(v, 97.5):+.4f}]{(v > 0).mean():>10.0%}")

comparison                                     mean                95% CI  better in
Model 4 - Model 3, same settings (A/B)      +0.0028   [+0.0016, +0.0039]      100%
Model 4 - Model 3, both tuned on 2023       +0.0050   [+0.0035, +0.0062]      100%
Model 4 (2023-tuned) - Model 3 shipped      +0.0018   [+0.0004, +0.0034]       99%


### EXPLORATION — How much does the model lean on the blurb?
Scramble `blurb_score` into noise, re-score 2024, see how far AUC falls. Compare with Model 3's ranking: subcategory 0.151, goal 0.036, prep 0.022, duration 0.019.

In [10]:
drops = []
for seed in range(5):
    Xs = X4_val.copy()
    Xs["blurb_score"] = np.random.default_rng(seed).permutation(Xs["blurb_score"].to_numpy())
    drops.append(AUC_M4 - roc_auc_score(y_val, model4.predict_proba(Xs)[:, 1]))
print(f"AUC lost when blurb_score is scrambled: {np.mean(drops):.4f}  (5 shuffles, range {min(drops):.4f}-{max(drops):.4f})")

AUC lost when blurb_score is scrambled: 0.0175  (5 shuffles, range 0.0169-0.0180)


### PIPELINE — Coaching metrics (validation 2024)
Same three views as Models 1–3, plus one pre-declared capacity rule that matches the guide's finding ("call the campaigns the model is least sure about"): **the 2,000 campaigns whose score is closest to 50%**. It is declared before looking, not searched, so it needs no split-half correction.

In [11]:
COACH_CAPACITY = 2_000
RANDOM_NM_RATE = nm_val.mean()
ids = val["id"].to_numpy()

def lists(score):
    s = np.asarray(score, float)
    risky  = np.lexsort((ids, s))[:COACH_CAPACITY]                   # riskiest-first
    unsure = np.lexsort((ids, np.abs(s - 0.5)))[:COACH_CAPACITY]     # closest to 50%
    br = (s >= 0.4) & (s < 0.6)                                      # pre-declared 40-60% bracket
    return {"riskiest-first hit": nm_val[risky].mean(),
            "closest-to-50% hit": nm_val[unsure].mean(),
            "40-60% bracket size": int(br.sum()),
            "40-60% bracket hit": nm_val[br].mean() if br.any() else np.nan,
            "calibration (pts)": 100 * (s.mean() - y_val.mean())}

models = {"Model 1: lookup": score_m1, "Model 2: logistic": score_m2,
          "Model 3: boosting (shipped)": score_m3, "Model 3: tuned on 2023": s3_h,
          "Model 4: + blurb (tuned on 2023)": score_m4}
summary = pd.DataFrame({k: {"AUC": roc_auc_score(y_val, v), **lists(v)} for k, v in models.items()}).T
summary["closest-to-50% lift"] = summary["closest-to-50% hit"] / RANDOM_NM_RATE
summary["bracket lift"] = summary["40-60% bracket hit"] / RANDOM_NM_RATE
print(f"random near-miss rate {RANDOM_NM_RATE:.2%}   |   capacity {COACH_CAPACITY:,}\n")
fmt = {"AUC": "{:.4f}", "riskiest-first hit": "{:.2%}", "closest-to-50% hit": "{:.2%}",
       "40-60% bracket size": "{:,.0f}", "40-60% bracket hit": "{:.2%}", "calibration (pts)": "{:+.1f}",
       "closest-to-50% lift": "{:.2f}x", "bracket lift": "{:.2f}x"}
display(summary.style.format(fmt))

random near-miss rate 2.02%   |   capacity 2,000



,AUC,riskiest-first hit,closest-to-50% hit,40-60% bracket size,40-60% bracket hit,calibration (pts),closest-to-50% lift,bracket lift
Model 1: lookup,0.8067,1.85%,4.35%,"7,101",3.38%,+5.7,2.15x,1.67x
Model 2: logistic,0.8526,0.95%,4.25%,"2,542",4.21%,+6.4,2.10x,2.08x
Model 3: boosting (shipped),0.8603,1.10%,4.80%,"2,243",4.68%,+6.1,2.37x,2.31x
Model 3: tuned on 2023,0.8571,1.05%,4.70%,"2,101",4.66%,+6.3,2.32x,2.31x
Model 4: + blurb (tuned on 2023),0.8621,1.20%,4.90%,"1,972",4.97%,+6.7,2.42x,2.46x


### EXPLORATION — What the grader thinks a good blurb looks like
For a slide. Highest- and lowest-scored 2024 blurbs, with what actually happened. Read these before presenting them: if the top blurbs are all board games and the bottom all apps, the grader is mostly reading category, which the within-category table above will already have shown.

In [12]:
show = val.assign(blurb_score=bs_val)[["blurb_score", "subcategory", "success", "blurb"]]
pd.set_option("display.max_colwidth", 110)
print("HIGHEST blurb_score (2024)")
display(show.nlargest(8, "blurb_score"))
print("LOWEST blurb_score (2024)")
display(show.nsmallest(8, "blurb_score"))

HIGHEST blurb_score (2024)


,blurb_score,subcategory,success,blurb
14581,0.966860,Design / Product Design,1,The classic fountain pen re-engineered for the modern desk with a click-action and an innovative sealing m...
20729,0.963541,Games / Tabletop Games,1,The 2nd volume in a beautiful collection of handpicked ancient games that fit in your pocket. Featuring 4 ...
1044,0.963387,Games / Tabletop Games,1,"Goblin's Assembly: A Collection of 4 High-Quality Heroic Miniatures, Up to 14 STL Files with Stretch Goals"""
20609,0.962117,other,1,"The second album from Claire Cope's expanded 11 piece ensemble, featuring eight original compositions, ins..."
20629,0.961821,Music / Electronic Music,1,"30th anniversary, Double Gatefold Vinyl & CD. Remastered, Remixed and Reimagined versions of the seminal 1..."
1059,0.961809,Games / Tabletop Games,1,A 8mm Epic-scale Terrain + 28mm Cyberpunk Terrain set reprint. Cool stuff for Epic-scale and Cyberpunk tab...
177,0.960298,Games / Tabletop Games,1,Solve a clockwork puzzle & build fantastical contraptions with the help of 3 new tinkerers.
832,0.958207,Games / Tabletop Games,1,Tsukuyumi: Full Moon Down returns once more with all-new expansions and a full reprint.


LOWEST blurb_score (2024)


,blurb_score,subcategory,success,blurb
5744,0.013509,Technology / Apps,0,I am looking to create an social media app based on video and to help people connect. We lost connection a...
19391,0.021608,Technology / Apps,0,Mobile application to ease access to products with A.I in marketspaces and online stores
11122,0.024151,Technology / Apps,0,A solution to a major problem in the construction industry; my app connects people who work in trades with...
11015,0.027984,Technology / Apps,0,Social networking application that lets you share outfits and buy tagged products directly
10420,0.029202,Technology / Apps,0,Sports and athletics media sharing application.
9065,0.032471,Technology / Apps,0,Help create a Free app and website for anyone and everyone to connect to other with: ConnnectedMinds.
7755,0.040296,Technology / Apps,0,"We require start up capital for application development, IP, marketing, legal"
8984,0.044636,Technology / Apps,0,ActualList.com uses a community based platform to foster informed choices by sharing anonymous customer re...


### PIPELINE — Save
- `model4_val.parquet`: 2024 scores (`id`, `blurb_score`, `score_model4`, `success`, `near_miss`) for the comparison table and Step 5.
- `blurb_scores.parquet`: `blurb_score` for every 2019–25 campaign (`id`, `slice`, `blurb_score`). The 2025 rows let the final run build `X_test + blurb_score` without re-embedding.
- `model4.joblib` + `blurb_grader.joblib`: the fitted models, so the one-time 2025 run just loads and scores.
- `model4_grid.csv`: the 2023 inner-holdout tuning record for both feature sets.

In [13]:
import joblib

pd.DataFrame({"id": ids, "blurb_score": bs_val, "score_model4": score_m4,
              "success": y_val, "near_miss": nm_val}).to_parquet(f"{BASE}/model4_val.parquet", index=False)
pd.concat([pd.DataFrame({"id": train["id"].to_numpy(), "slice": "train", "blurb_score": bs_train}),
           pd.DataFrame({"id": ids, "slice": "val", "blurb_score": bs_val}),
           pd.DataFrame({"id": test_bl["id"].to_numpy(), "slice": "test", "blurb_score": bs_test})]
          ).to_parquet(f"{BASE}/blurb_scores.parquet", index=False)
grid.to_csv(f"{BASE}/model4_grid.csv", index=False)
joblib.dump(model4, f"{BASE}/model4.joblib")
joblib.dump(grader, f"{BASE}/blurb_grader.joblib")

OUT = ["model4_val.parquet", "blurb_scores.parquet", "model4_grid.csv", "model4.joblib", "blurb_grader.joblib"]
print("saved:", ", ".join(OUT))
try:
    from google.colab import files
    os.system(f"cd {BASE} && zip -q model4_outputs.zip " + " ".join(OUT))
    files.download(f"{BASE}/model4_outputs.zip")
except ImportError:
    pass

saved: model4_val.parquet, blurb_scores.parquet, model4_grid.csv, model4.joblib, blurb_grader.joblib


## Model 4 summary *(fill in after running)*

| Metric (validation 2024) | Model 3 (shipped) | Model 3 (tuned on 2023) | **Model 4** |
|---|---|---|---|
| AUC | 0.860 | 0.857 | **___** |
| Gain vs Model 3, same settings (A/B) | | | ___ [CI ___] |
| Gain vs Model 3, both tuned on 2023 | | | ___ [CI ___] |
| Blurb grader alone | | | ___ (category lookup: 0.807) |
| AUC lost if `blurb_score` scrambled | | | ___ |
| 40–60% bracket | 4.68% (2,243) | ___ | ___ |
| Closest-to-50%, 2,000 calls | ___ | ___ | ___ |
| Calibration | +6.1 pts | ___ | ___ |

**How to read it:** if the A/B gain's CI excludes zero, the blurb adds real information beyond category, goal and timing. Then say how much in plain terms. If the gain is ~0 and the grader alone scores high, the blurb mostly re-tells the category: a finding, not a failure ("what you write matters less than what you make and how much you ask for").

**Pick for Step 5:** the model with the better *coaching* number, with AUC as tie-breaker. Then run 2025 once.

**Limitations:** frozen small encoder (no fine-tuning); grader trained on 2016–18 when success was 54%, so its raw scores run low on later years (harmless for trees, stated anyway); blurbs only, not titles, images or videos.

In [14]:
print(summary.to_string(float_format=lambda x: f"{x:.4f}"))

                                    AUC  riskiest-first hit  closest-to-50% hit  40-60% bracket size  40-60% bracket hit  calibration (pts)  closest-to-50% lift  bracket lift
Model 1: lookup                  0.8067              0.0185              0.0435            7101.0000              0.0338             5.7286               2.1500        1.6705
Model 2: logistic                0.8526              0.0095              0.0425            2542.0000              0.0421             6.3578               2.1006        2.0804
Model 3: boosting (shipped)      0.8603              0.0110              0.0480            2243.0000              0.0468             6.1257               2.3724        2.3137
Model 3: tuned on 2023           0.8571              0.0105              0.0470            2101.0000              0.0466             6.2887               2.3230        2.3054
Model 4: + blurb (tuned on 2023) 0.8621              0.0120              0.0490            1972.0000              0.0497     